# Conversion DWG → DXF (LibreDWG)

Conversion d'un fichier DWG en DXF via **LibreDWG** (dwg2dxf), sans ODA.

**Prérequis :** télécharger les binaires Windows LibreDWG, les extraire, puis :
- ajouter le dossier au PATH, ou
- définir la variable d'environnement `LIBREDWG_DIR`

Téléchargement : [LibreDWG releases](https://github.com/LibreDWG/libredwg/releases) (asset : `libredwg-<version>-win64.zip`)


In [ ]:
# --- Imports et configuration ---
import logging
import os
import shutil
import subprocess
from pathlib import Path

LIBREDWG_DIR = os.environ.get("LIBREDWG_DIR", r"C:\Users\mvm\libredwg")
CONVERT_TIMEOUT = 120  # secondes (augmenter pour très gros DWG)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

In [ ]:
# --- Fonctions ---

def _find_dwg2dxf() -> str | None:
    """Cherche dwg2dxf dans PATH ou LIBREDWG_DIR. Retourne le chemin ou None."""
    exe = "dwg2dxf.exe" if os.name == "nt" else "dwg2dxf"
    if path := shutil.which(exe):
        return path
    for sub in ("", "bin"):
        candidate = Path(LIBREDWG_DIR) / sub / exe
        if candidate.is_file():
            return str(candidate)
    return None


def convert_dwg_to_dxf(
    dwg_path: str,
    dxf_path: str,
    *,
    dwg2dxf_path: str | None = None,
    overwrite: bool = True,
    timeout: int = CONVERT_TIMEOUT,
) -> None:
    """Convertit un fichier DWG en DXF via LibreDWG (dwg2dxf)."""
    src = Path(dwg_path).resolve()
    dest = Path(dxf_path).resolve()

    if not src.is_file():
        raise FileNotFoundError(f"Fichier source introuvable : {src}")
    if src.stat().st_size == 0:
        raise ValueError(f"Fichier source vide : {src}")

    exe = dwg2dxf_path or _find_dwg2dxf()
    if not exe or not Path(exe).is_file():
        raise FileNotFoundError(
            "dwg2dxf introuvable. Installez LibreDWG : "
            "https://github.com/LibreDWG/libredwg/releases"
        )

    dest.parent.mkdir(parents=True, exist_ok=True)
    cmd = [exe, "-y" if overwrite else None, "-o", str(dest), str(src)]
    cmd = [c for c in cmd if c is not None]

    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=timeout,
            creationflags=subprocess.CREATE_NO_WINDOW if os.name == "nt" else 0,
        )
    except subprocess.TimeoutExpired:
        raise RuntimeError(f"Timeout ({timeout}s) dépassé pour {src.name}")
    except OSError as e:
        raise RuntimeError(f"Impossible d'exécuter dwg2dxf : {e}") from e

    if result.returncode != 0:
        err = (result.stderr or result.stdout or "").strip()
        raise RuntimeError(f"dwg2dxf a échoué (code {result.returncode}) : {err or 'aucune sortie'}")

    if not dest.is_file() or dest.stat().st_size == 0:
        raise RuntimeError(f"Fichier DXF non créé ou vide : {dest}")

In [ ]:
# --- Conversion (modifier les chemins) ---
DWG_PATH = r"C:\Users\mvm\Geolux_CV_Clone\exemple.dwg"
DXF_PATH: str | None = None  # None = même dossier que le DWG, extension .dxf

dwg_file = Path(DWG_PATH).resolve()
dxf_file = Path(DXF_PATH).resolve() if DXF_PATH else dwg_file.with_suffix(".dxf")

convert_dwg_to_dxf(str(dwg_file), str(dxf_file), overwrite=True)
logger.info("Conversion terminée : %s → %s", dwg_file.name, dxf_file.name)

In [ ]:
import logging
import ezdxf
import pandas as pd
from pathlib import Path

def _safe_utf8(s):
    """Enlève les caractères surrogate / invalides UTF-8 qui provoquent UnicodeEncodeError."""
    if not isinstance(s, str):
        return s
    return s.encode("utf-8", errors="replace").decode("utf-8")

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger(__name__)

dxfs_dir = Path(r"C:\Users\mvm\Geolux_CV_Clone\dxf_test")
if not dxfs_dir.is_dir():
    raise FileNotFoundError(f"Dossier introuvable : {dxfs_dir}")

dxf_files = sorted(dxfs_dir.glob("*.dxf"), key=lambda p: p.name.lower())
total_files = len(dxf_files)
logger.info("Début lecture DXF : %d fichier(s) dans %s", total_files, dxfs_dir)

all_rows = []
failed_files = []

for i, path in enumerate(dxf_files, start=1):
    name = path.name
    try:
        doc = ezdxf.readfile(str(path))
        msp = doc.modelspace()
        count = 0
        for e in msp:
            row = {
                "file_name": _safe_utf8(name),
                "entity_type": _safe_utf8(e.dxftype()),
                "handle": getattr(e.dxf, "handle", None),
            }
            for key, value in e.dxfattribs().items():
                if value is None or isinstance(value, (int, float, bool)):
                    row[key] = value
                elif isinstance(value, str):
                    row[key] = _safe_utf8(value)
                else:
                    row[key] = _safe_utf8(str(value))
            all_rows.append(row)
            count += 1
        logger.info("[%d/%d] %s — OK (%d entités)", i, total_files, _safe_utf8(name), count)
    except Exception as ex:
        failed_files.append((_safe_utf8(name), _safe_utf8(str(ex))))
        logger.error("[%d/%d] %s — Échec : %s", i, total_files, _safe_utf8(name), _safe_utf8(str(ex)))

df_origin = pd.DataFrame(all_rows)
logger.info(
    "Terminé : %d fichier(s) lu(s), %d entité(s), %d échec(s)%s",
    total_files - len(failed_files),
    len(all_rows),
    len(failed_files),
    f" — {[f[0] for f in failed_files]}" if failed_files else "",
)